## Build dim_product table in BigQuery

In [1]:
from google.cloud import bigquery

client = bigquery.Client()
project_id = "linear-theater-436300-r9"
dataset_id = "ecommerce_pipeline"

# dim_product will simply be a cleaned version of products_cleaned
dim_product_table = f"{project_id}.{dataset_id}.dim_product"

query = f"""
CREATE OR REPLACE TABLE `{dim_product_table}` AS
SELECT
    item_id,
    category,
    brand,
    title,
    description,
    price,
    price_segment,
    title_length,
    description_length,
    metadata_length
FROM `{project_id}.{dataset_id}.products_cleaned`
"""

job = client.query(query)
job.result()

print("dim_product table created successfully.")


dim_product table created successfully.


## Build dim_time table in BigQuery

In [2]:
from google.cloud import bigquery

client = bigquery.Client()
project_id = "linear-theater-436300-r9"
dataset_id = "ecommerce_pipeline"

dim_time_table = f"{project_id}.{dataset_id}.dim_time"

query = f"""
CREATE OR REPLACE TABLE `{dim_time_table}` AS
WITH dates AS (
  SELECT
    date AS full_date,
    EXTRACT(YEAR FROM date) AS year,
    EXTRACT(MONTH FROM date) AS month,
    EXTRACT(DAY FROM date) AS day,
    EXTRACT(DAYOFWEEK FROM date) AS day_of_week
  FROM UNNEST(
    GENERATE_DATE_ARRAY('2020-01-01', '2024-12-31', INTERVAL 1 DAY)
  ) AS date
)
SELECT
    FORMAT_DATE('%Y%m%d', full_date) AS time_id,
    full_date,
    year,
    month,
    day,
    day_of_week
FROM dates
ORDER BY full_date
"""

job = client.query(query)
job.result()

print("dim_time table created successfully.")


dim_time table created successfully.


## Build fact_reviews table in BigQuery

In [3]:
from google.cloud import bigquery

client = bigquery.Client()
project_id = "linear-theater-436300-r9"
dataset_id = "ecommerce_pipeline"

fact_reviews_table = f"{project_id}.{dataset_id}.fact_reviews"

query = f"""
CREATE OR REPLACE TABLE `{fact_reviews_table}` AS
SELECT
    GENERATE_UUID() AS review_id,
    content_clean,
    label,
    sentiment_strength,
    content_length,
    word_count
FROM `{project_id}.{dataset_id}.reviews_cleaned`
"""

job = client.query(query)
job.result()

print("fact_reviews table created successfully.")


fact_reviews table created successfully.


## Warehouse verification

In [4]:
from google.cloud import bigquery

client = bigquery.Client()
project_id = "linear-theater-436300-r9"
dataset_id = "ecommerce_pipeline"

# 1. Validate dim_product
q1 = f"SELECT COUNT(*) AS row_count FROM `{project_id}.{dataset_id}.dim_product`"
dim_product_count = client.query(q1).to_dataframe()

# 2. Validate dim_time
q2 = f"SELECT COUNT(*) AS row_count FROM `{project_id}.{dataset_id}.dim_time`"
dim_time_count = client.query(q2).to_dataframe()

# 3. Validate fact_reviews
q3 = f"SELECT COUNT(*) AS row_count FROM `{project_id}.{dataset_id}.fact_reviews`"
fact_reviews_count = client.query(q3).to_dataframe()

print("dim_product rows:", dim_product_count.iloc[0,0])
print("dim_time rows:", dim_time_count.iloc[0,0])
print("fact_reviews rows:", fact_reviews_count.iloc[0,0])


/home/niranjanrao07/cod-multiagent-ecommerce/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


dim_product rows: 1048018
dim_time rows: 1827
fact_reviews rows: 50000
